# Train LSTM — BISINDO Sequence Classifier (Google Colab)

Notebook untuk melatih model LSTM yang mengklasifikasikan urutan landmark MediaPipe menjadi label kata BISINDO.

**Pipeline:**
1. Setup environment (TensorFlow + MediaPipe)
2. Mount Drive + clone repo
3. Load dataset landmark (.npy) dari Drive
4. Normalisasi + augmentasi
5. Training LSTM
6. Evaluasi: confusion matrix, classification report
7. Export `bisindo_lstm.h5` + `labels.json` ke Drive

## 1. Setup

In [1]:
!nvidia-smi || echo 'No GPU detected — training akan jalan di CPU (lambat).'

Tue May  5 16:56:55 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   46C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# Versi disamakan dengan requirements.txt repo (TF 2.21, MediaPipe 0.10.35, sklearn 1.8).
!pip install -q tensorflow==2.20.0 mediapipe==0.10.35 scikit-learn==1.8.0 seaborn==0.13.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 620.7/620.7 MB 1.4 MB/s eta 0:00:00:00:0100:01


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [9]:
# Clone project repo agar bisa import app.services.landmark_normalizer / landmark_augmenter.
# Ganti URL dengan repo Anda. Kalau repo private: pakai PAT atau ssh-key.
REPO_URL  = 'https://github.com/Suhardianto-Rimanda/sibindo-translator.git'
REPO_PATH = '/content/sibindo'

import os, sys
if not os.path.exists(REPO_PATH):
    !git clone $REPO_URL $REPO_PATH
sys.path.insert(0, REPO_PATH)
print('repo path:', REPO_PATH)

repo path: /content/sibindo


## 2. Konfigurasi

In [ ]:
PROJECT_DIR = '/content/drive/MyDrive/sibindo'
DATA_DIR    = f'{PROJECT_DIR}/datasets/landmarks'   # subfolder per label, isi .npy
WEIGHTS_OUT = f'{PROJECT_DIR}/weights/bisindo_lstm.h5'
LABELS_OUT  = f'{PROJECT_DIR}/weights/labels.json'
HISTORY_OUT = f'{PROJECT_DIR}/weights/training_history.json'

SEQ_LEN     = 30           # harus sama dengan saat collect_landmarks.py
FEAT_DIM    = 126          # 21 lm * 3 dim * 2 hands
EPOCHS      = 100
BATCH       = 16
AUG_FACTOR  = 4
LR          = 1e-3
SEED        = 42

import os
os.makedirs(f'{PROJECT_DIR}/weights', exist_ok=True)

## 3. Load + Normalize Dataset

In [11]:
from pathlib import Path
import numpy as np
from app.services.landmark_normalizer import normalize_sequence
from app.services.landmark_augmenter import augment

np.random.seed(SEED)

data_dir = Path(DATA_DIR)
assert data_dir.exists(), f'DATA_DIR tidak ditemukan: {data_dir}'
labels = sorted([d.name for d in data_dir.iterdir() if d.is_dir()])
assert labels, f'Tidak ada subfolder label di {data_dir}'
print('Labels:', labels)

X, y, skipped = [], [], 0
for idx, label in enumerate(labels):
    for npy in sorted((data_dir / label).glob('*.npy')):
        seq = np.load(npy)
        if seq.shape != (SEQ_LEN, FEAT_DIM):
            skipped += 1
            continue
        X.append(normalize_sequence(seq))
        y.append(idx)

X = np.array(X, dtype=np.float32)
y = np.array(y)
print('Shape:', X.shape, '| classes:', len(labels), '| skipped:', skipped)
print('Per-class counts:', dict(zip(*np.unique(y, return_counts=True))))

ModuleNotFoundError: No module named 'flask_cors'

## 4. Split + Augment

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.utils import to_categorical

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.15, random_state=SEED, stratify=y_train
)
print('train:', len(X_train), '| val:', len(X_val), '| test:', len(X_test))

# Augment train only
aug_X = list(X_train)
aug_y = list(y_train)
for _ in range(AUG_FACTOR):
    for seq, lbl in zip(X_train, y_train):
        aug_X.append(augment(seq))
        aug_y.append(lbl)
X_train = np.array(aug_X, dtype=np.float32)
y_train = np.array(aug_y)
print('after augment train:', len(X_train))

y_train_cat = to_categorical(y_train, num_classes=len(labels))
y_val_cat   = to_categorical(y_val,   num_classes=len(labels))
y_test_cat  = to_categorical(y_test,  num_classes=len(labels))

cw = compute_class_weight('balanced', classes=np.arange(len(labels)), y=y_train)
cw_dict = {i: float(w) for i, w in enumerate(cw)}
print('class weights:', cw_dict)

## 5. Build + Train LSTM

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

model = Sequential([
    Input(shape=(SEQ_LEN, FEAT_DIM)),
    LSTM(64, return_sequences=True, activation='tanh'),
    BatchNormalization(),
    Dropout(0.3),
    LSTM(128, return_sequences=False, activation='tanh'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(len(labels), activation='softmax'),
])
model.compile(optimizer=Adam(learning_rate=LR),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True),
    ModelCheckpoint(WEIGHTS_OUT, monitor='val_accuracy', save_best_only=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=8, min_lr=1e-5),
]

history = model.fit(
    X_train, y_train_cat,
    validation_data=(X_val, y_val_cat),
    epochs=EPOCHS,
    batch_size=BATCH,
    callbacks=callbacks,
    class_weight=cw_dict,
)

## 6. Evaluasi

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history['accuracy'],     label='train')
ax[0].plot(history.history['val_accuracy'], label='val')
ax[0].set_title('Accuracy'); ax[0].legend(); ax[0].grid(alpha=0.3)
ax[1].plot(history.history['loss'],     label='train')
ax[1].plot(history.history['val_loss'], label='val')
ax[1].set_title('Loss'); ax[1].legend(); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

y_pred = model.predict(X_test, verbose=0).argmax(axis=1)
print(classification_report(y_test, y_pred, target_names=labels, zero_division=0))

cm = confusion_matrix(y_test, y_pred)
side = max(6, len(labels) * 0.6)
plt.figure(figsize=(side, side))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels)
plt.title('Confusion Matrix'); plt.xlabel('Predicted'); plt.ylabel('True')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

## 7. Export ke Drive

In [ ]:
import json

# EarlyStopping(restore_best_weights=True) sudah me-restore bobot terbaik
# ke `model` sebelum fit() return, jadi cukup save sekali di sini.
model.save(WEIGHTS_OUT)
print('saved model:', WEIGHTS_OUT)

with open(LABELS_OUT, 'w', encoding='utf-8') as f:
    json.dump(labels, f, ensure_ascii=False, indent=2)
print('saved labels:', LABELS_OUT)

with open(HISTORY_OUT, 'w', encoding='utf-8') as f:
    json.dump({k: list(map(float, v)) for k, v in history.history.items()},
              f, indent=2)
print('saved history:', HISTORY_OUT)